# GPU worker trên Colab

Notebook chạy Celery worker của `ai_workers` trên GPU Colab và nhận job từ website qua queue `gpu`.
Website tự chuyển job sang worker CPU local khi notebook này tắt.

Trước khi chạy, thêm vào **Colab Secrets** (biểu tượng chìa khóa bên trái) và bật quyền truy cập notebook:

| Secret | Bắt buộc | Ghi chú |
|---|---|---|
| `SHARED_REDIS_URL` | có | Cùng Redis với backend, ví dụ `rediss://default:<pass>@<host>:6379?ssl_cert_reqs=CERT_REQUIRED` |
| `CF_R2_ACCOUNT_ID`, `CF_R2_ACCESS_KEY_ID`, `CF_R2_SECRET_ACCESS_KEY`, `CF_R2_BUCKET_NAME`, `CF_R2_PUBLIC_URL` | có | Worker tải video và upload keyframe qua R2 |
| `GROQ_API_KEY` | có | Tóm tắt LLM |
| `OPENROUTER_API_KEY`, `HF_TOKEN` | không | Fallback LLM, pyannote |
| `GITHUB_TOKEN` | nếu repo private | Token chỉ có quyền đọc repo |

Runtime: **Runtime → Change runtime type → GPU** (T4 trở lên). Hướng dẫn đầy đủ: `docs/colab-gpu-worker-guide.md`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Cache model trên Drive để session sau không tải lại (~3 GB). Bỏ qua cell này nếu không cần.
from google.colab import drive
drive.mount('/content/drive')
CACHE_DIR = '/content/drive/MyDrive/mls-cache'

In [ ]:
import os, subprocess
from google.colab import userdata

REPO_URL = 'https://github.com/multimodal-lecture-summarizer/multimodal-lecture-summarizer.git'
BRANCH = 'feat/update-project'
REPO_DIR = '/content/mls'

try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = REPO_URL.replace('https://', f'https://x-access-token:{token}@') if token else REPO_URL

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO_DIR], check=True)
# Florence-2 weights live in Git LFS; the worker refuses to start without them.
subprocess.run('git lfs install && git lfs pull', shell=True, cwd=REPO_DIR, check=True)

In [ ]:
# Python 3.11 + the same requirement files as ai_workers/Dockerfile.
!pip install -q uv
!uv python install 3.11
!uv venv --python 3.11 /content/venv
!cd /content/mls && VIRTUAL_ENV=/content/venv uv pip install -r ai_workers/requirements.txt -r backend/requirements.txt

In [ ]:
import os
from google.colab import userdata

def secret(name, required=True):
    try:
        return userdata.get(name)
    except Exception:
        if required:
            raise RuntimeError(f'Thiếu Colab Secret: {name}')
        return ''

redis_url = secret('SHARED_REDIS_URL')
cache_dir = globals().get('CACHE_DIR', '/content/mls/cache')
site = '/content/venv/lib/python3.11/site-packages'

worker_env = {
    **os.environ,
    'CELERY_BROKER_URL': redis_url,
    'CELERY_RESULT_BACKEND': redis_url,
    'CF_R2_ACCOUNT_ID': secret('CF_R2_ACCOUNT_ID'),
    'CF_R2_ACCESS_KEY_ID': secret('CF_R2_ACCESS_KEY_ID'),
    'CF_R2_SECRET_ACCESS_KEY': secret('CF_R2_SECRET_ACCESS_KEY'),
    'CF_R2_BUCKET_NAME': secret('CF_R2_BUCKET_NAME'),
    'CF_R2_PUBLIC_URL': secret('CF_R2_PUBLIC_URL'),
    'GROQ_API_KEY': secret('GROQ_API_KEY'),
    'OPENROUTER_API_KEY': secret('OPENROUTER_API_KEY', required=False),
    'HF_TOKEN': secret('HF_TOKEN', required=False),
    'CACHE_DIR': cache_dir,
    'HF_HOME': f'{cache_dir}/huggingface',
    'FLORENCE_DEVICE': 'cuda',
    'SEMANTIC_CLIP_DEVICE': 'cuda',
    'PADDLEOCR_USE_GPU': '0',
    'CUBLAS_WORKSPACE_CONFIG': ':4096:8',
    'PROCESS_MIN_AVAILABLE_MEMORY_MB': '4096',
    'PROCESS_SOFT_MIN_AVAILABLE_MEMORY_MB': '3072',
    # Faster-Whisper loads cuDNN 9 / cuBLAS from the torch wheels.
    'LD_LIBRARY_PATH': f'{site}/nvidia/cudnn/lib:{site}/nvidia/cublas/lib:' + os.environ.get('LD_LIBRARY_PATH', ''),
    'PATH': '/content/venv/bin:' + os.environ['PATH'],
}
os.makedirs(cache_dir, exist_ok=True)
print('Cấu hình xong. Cache:', cache_dir)

In [ ]:
# Chạy worker. Giữ cell này chạy; dừng cell = worker rời queue, website quay về worker CPU.
import subprocess
proc = subprocess.Popen(
    ['celery', '-A', 'ai_workers.core.celery_app', 'worker', '--loglevel=info',
     '--pool=solo', '--concurrency=1', '-Q', 'gpu', '-n', 'colab@%h'],
    cwd='/content/mls', env=worker_env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
try:
    for line in proc.stdout:
        print(line, end='')
except KeyboardInterrupt:
    proc.terminate()
    proc.wait()
    print('Worker đã dừng.')